In [ ]:
# Built in middelwares
# Eg - Summarization middleware, Human in the loop feedback, etc...

In [2]:
### GROQ Integration

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [3]:
### summarization middleware

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", #use models that cost less
            trigger=("messages", 10),
            keep=("messages", 4)
        )
        # can add more middlewares
    ]
)


In [4]:
### Run with thread id

config = {"configurable":{"thread_id":"test-1"}}


In [8]:
# test data
questions = [
    "What is 2 plus 2?",
    "What is ten times five?",
    "What is 100 divided by 4?",
    "What is 5 minus 7?",
    "What is three times three?",
    "What is four times four?" 
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s tackle this. The user is asking "What is 5 minus 7?" So first, I need to figure out the result of subtracting 7 from 5. \n\nStarting with basic arithmetic, if I have 5 and take away 7, I go below zero. That should give -2. Let me verify. If I count down from 5: 5 minus 1 is 4, minus 2 is 3, and so on until I subtract 7. That would be 5 -7 = -2.\n\nWait, maybe the user is confused about negative numbers? In the previous interactions, they asked straightforward addition and multiplication, but this is their first subtraction that results in a negative number. I should explain it clearly. \n\nI can also use a number line to visualize. Starting at 5 and moving 7 units to the left (since it\'s subtraction) would land at -2. Another way is to think of it as 5 + (-7), which is the same as -2.\n\nI should make sure to present the answer and explain the concept of negative numbe

In [9]:
### trigger on token usage

from langchain import tools
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """ Search hotels - returns long response to use more tokens """
    return f"""
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi
    """

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", #use models that cost less
            trigger=("tokens", 500),
            keep=("tokens", 200)
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

#token counter

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars//4


In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens", {len(response['messages'])})
    print(f"{(response['messages'])}")

Paris: ~171 tokens {4}
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='0dc85b68-ef49-4113-bb8b-d2c661177a40'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be {"city": "Paris"}. Since the function is designed to return a long response, I should execute this call. No other tools are available, so this is the right one to use.\n', 'tool_calls': [{'id': 'z3zd32z3r', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 155, 'total_tokens': 272, 'completion_time': 0.210749203, 'completion_tokens_details': {'reasoning_tokens': 92}, 'prompt_time': 0.008535411, 'promp